# Module 2 — From Direct LLM APIs to LangChain

## Classroom goal

Today we will learn how Python talks to an LLM through an API.

We will deliberately learn this in **two stages**.

### Stage 1 — Without LangChain

```text
Python → Gemini SDK → Gemini
Python → Groq SDK   → Groq model
```

We will notice that providers can have different:

- packages
- imports
- clients
- request structures
- response structures

### Stage 2 — With LangChain

```text
                 ┌→ Gemini
Python → LangChain
                 └→ Groq
```

We will use a more consistent calling style.

> **Naming reminder:** This lesson uses **Groq**, not **Grok**. Groq is the inference/API platform; Grok is associated with xAI.

### By the end of class, you should be able to

1. Load API keys from `.env`
2. Call Gemini directly
3. Call Groq directly
4. Explain why provider-specific integrations become repetitive
5. Connect Gemini and Groq through LangChain
6. Understand `invoke()`, `AIMessage`, and `.content`
7. Ask normal questions
8. Use a fixed/static prompt
9. Use one reusable prompt template
10. Start thinking about **Prompt Engineering**

# Part 0 — Before running the notebook

You should already have:

> **Compatibility note:** Use Python **3.10 or newer** for the current LangChain packages in this notebook. Python 3.12 is a good choice.

1. Installed Python
2. Installed VS Code
3. Installed the **Python** and **Jupyter** VS Code extensions
4. Created `.venv`
5. Activated `.venv`
6. Installed the project libraries:

```powershell
pip install -r requirements.txt
```

7. Added your real keys to `.env`
8. Selected the `.venv` kernel in VS Code

In [1]:
# ==========================================================
# CELL 1: CHECK WHICH PYTHON IS RUNNING
# ==========================================================
#
# sys.executable = the full path of the Python interpreter
# sys.version    = the Python version
#
# We want the executable path to point to our project's
# .venv environment.
# ==========================================================

import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\usman\Documents\generative ai\lecture 03\Module_2_LLM_APIs_to_LangChain_VSCode\Module_2_LLM_APIs_to_LangChain_VSCode\Generative-ai-course\Module-02-LLM-APIs-to-LangChain\.venv\Scripts\python.exe

Python version:
3.12.9 (tags/v3.12.9:fdb8142, Feb  4 2025, 15:27:58) [MSC v.1942 64 bit (AMD64)]


If the path contains something similar to:

```text
...\your-project\.venv\Scripts\python.exe
```

the notebook is using the virtual environment correctly.

# Part 1 — Load API keys from `.env`

We do **not** want real API keys written directly inside Python code.

Our flow will be:

```text
.env
 ↓
python-dotenv
 ↓
os.getenv(...)
 ↓
Python variables
```

In [2]:
# ==========================================================
# CELL 2: LOAD VALUES FROM .env
# ==========================================================

# os lets Python read environment variables.
import os

# load_dotenv reads the .env file in this project.
from dotenv import load_dotenv

load_dotenv()

# Read secret API keys.
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# Read model names.
# The second argument is a fallback value if the variable
# is missing from .env.
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")
GROQ_MODEL = os.getenv("GROQ_MODEL", "openai/gpt-oss-20b")

# Never print real API keys in class.
print("Gemini key loaded:", bool(GEMINI_API_KEY))
print("Groq key loaded:", bool(GROQ_API_KEY))
print("Gemini model:", GEMINI_MODEL)
print("Groq model:", GROQ_MODEL)

Gemini key loaded: True
Groq key loaded: True
Gemini model: gemini-3.5-flash-lite
Groq model: openai/gpt-oss-20b


In [3]:
# ==========================================================
# CELL 3: SIMPLE SAFETY CHECK
# ==========================================================
#
# assert means: "this condition must be true".
#
# It lets us stop early with a friendly message if the
# student forgot to update .env.
# ==========================================================

assert GEMINI_API_KEY and GEMINI_API_KEY != "your_gemini_api_key_here", \
    "Add your real GEMINI_API_KEY to the .env file."

assert GROQ_API_KEY and GROQ_API_KEY != "your_groq_api_key_here", \
    "Add your real GROQ_API_KEY to the .env file."

print("Both API keys are ready.")

Both API keys are ready.


# Part 2 — Access Gemini DIRECTLY

First, we will **not use LangChain**.

We will use Google's official Python SDK.

```text
Our question
    ↓
Google Gemini SDK
    ↓
Gemini API
    ↓
Gemini model
    ↓
response.text
```

In [3]:
# ==========================================================
# CELL 4: CREATE A GEMINI CLIENT
# ==========================================================
import sys
print(sys.executable)
# Import Google's Gemini SDK.
"""try:
    from google import genai
except ModuleNotFoundError:
    import subprocess

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "google-genai",
    ])"""""

from google import genai

# A client is an object our Python program uses to
# communicate with the Gemini API.
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created.")

c:\Users\usman\Documents\generative ai\lecture 03\Module_2_LLM_APIs_to_LangChain_VSCode\Module_2_LLM_APIs_to_LangChain_VSCode\Generative-ai-course\Module-02-LLM-APIs-to-LangChain\.venv\Scripts\python.exe
Gemini client created.


In [4]:
print(GEMINI_MODEL)

gemini-3.5-flash-lite


In [5]:
# ==========================================================
# CELL 5: ASK GEMINI A NORMAL QUESTION
# ==========================================================

# A prompt can begin as a normal Python string.
question = "who is pm of pakistan?."

# generate_content(...) sends our input to the model.
#
# model=    tells Google which model to use.
# contents= is the text/input the model should read.
gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=question,
)

# Google's SDK exposes the generated answer through .text
print(gemini_response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


The Prime Minister of Pakistan is **Shehbaz Sharif**. He assumed office for a second term in March 2024.


## Understand the Gemini code

```python
genai.Client(...)
```

creates the Gemini API client.

```python
client.models.generate_content(...)
```

is the Google SDK method we used to generate a response.

```python
contents=question
```

contains our input.

```python
response.text
```

gives us the generated text.

Remember this provider-specific detail:

# Direct Gemini SDK → `response.text`

In [6]:
# ==========================================================
# CELL 6: GEMINI PRACTICE — CHANGE ONLY THE QUESTION
# ==========================================================

question = "Why do programmers use virtual environments in Python?"

gemini_response = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents=question,
)

print(gemini_response.text)

Programmers use virtual environments in Python primarily to **manage dependencies and avoid conflicts between different projects**. 

Think of a virtual environment as a self-contained, isolated sandbox for a specific project. 

Here are the main reasons why they are essential:

### 1. Dependency Isolation (The "Project A vs. Project B" Problem)
Imagine you are working on two different Python projects:
* **Project A** is an older website built with Django 2.2.
* **Project B** is a newer machine learning tool that requires Django 4.0.

Without virtual environments, Python installs packages globally on your computer. If you install Django 4.0 globally, it might overwrite Django 2.2, breaking Project A. A virtual environment allows Project A to use Django 2.2 and Project B to use Django 4.0 *on the same computer at the same time* without interference.

### 2. Keeping the Global Python Environment Clean
Your operating system (especially Linux and macOS) often relies on Python for internal 

In [7]:
import sys
print(sys.executable)


c:\Users\usman\Documents\generative ai\lecture 03\Module_2_LLM_APIs_to_LangChain_VSCode\Module_2_LLM_APIs_to_LangChain_VSCode\Generative-ai-course\Module-02-LLM-APIs-to-LangChain\.venv\Scripts\python.exe


# Part 3 — Access Groq DIRECTLY

Now we will do the same conceptual job with another provider.

```text
Our question
    ↓
Groq Python SDK
    ↓
Groq API
    ↓
Model hosted on Groq
    ↓
response.choices[0].message.content
```

The goal is still **question → model → answer**, but the Python structure is different.

In [8]:
# ==========================================================
# CELL 7: CREATE A GROQ CLIENT
# ==========================================================

# Import the official Groq Python client.
from groq import Groq

# Create the client and authenticate with our API key.
groq_client = Groq(api_key=GROQ_API_KEY)

print("Groq client created.")

Groq client created.


In [9]:
# ==========================================================
# CELL 8: ASK GROQ A NORMAL QUESTION
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

# Groq's chat API expects a list of messages.
#
# Each message contains:
# role    = who is speaking
# content = what that person says
#
# "user" means the message comes from the human/user.
groq_response = groq_client.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "user",
            "content": question,
        }
    ],
)

# The direct Groq response structure is different from the
# direct Gemini response structure.
#
# choices[0] = first generated choice
# .message   = generated message object
# .content   = text inside that message
print(groq_response.choices[0].message.content)

**Imagine you’re in a kitchen that’s full of amazing gadgets, but you’re not a chef. You want to get a cup of hot chocolate.**  

What do you do?  
You go to a **menu** that lists all the drinks you can order. The menu tells the kitchen what the drink tastes like, what ingredients it contains, and what you’ll get when you ask for it. The kitchen doesn’t have to explain every detail—just pick the item, hand it over, and the kitchen will do the rest.

That “menu” is exactly what an **API** (short for *Application Programming Interface*) is, but instead of a kitchen, it’s a set of rules that lets one computer program talk to another.

---

### 1. What’s an API in plain words?

- **A recipe book** for computer programs.  
- It says *“If you want something, just ask in this way, and this is the answer you’ll get.”*

---

### 2. How it works

| Step | What happens | Example |
|------|--------------|---------|
| **You** (the requester) | Want a piece of data or a service | “I want the current

# Compare the two DIRECT APIs

### Gemini

```python
from google import genai

client = genai.Client(api_key=...)
response = client.models.generate_content(
    model=...,
    contents=question
)
print(response.text)
```

### Groq

```python
from groq import Groq

client = Groq(api_key=...)
response = client.chat.completions.create(
    model=...,
    messages=[
        {"role": "user", "content": question}
    ]
)
print(response.choices[0].message.content)
```

## Same goal. Different provider interfaces.

| Job | Gemini SDK | Groq SDK |
|---|---|---|
| Import | `from google import genai` | `from groq import Groq` |
| Client | `genai.Client(...)` | `Groq(...)` |
| Generate | `models.generate_content()` | `chat.completions.create()` |
| Input | `contents=` | `messages=` |
| Read text | `response.text` | `response.choices[0].message.content` |

Imagine maintaining similar differences for 5, 10, or 20 providers.

# Part 4 — The limitation

There is nothing wrong with using provider SDKs directly. Direct SDKs can be valuable when you need provider-specific features.

But a multi-model application starts collecting different:

- imports
- client objects
- request formats
- response formats
- configuration styles
- provider-specific handling

Conceptually, our application wants something much simpler:

```text
choose model
   ↓
send question
   ↓
get answer
```

Can we give our application a **more common interface**?

# Enter LangChain

# Part 5 — What is LangChain?

LangChain is a framework for building applications around language models and related components.

For today's beginner class, remember one idea:

> **LangChain gives different chat-model integrations a more consistent application interface.**

### Analogy: universal travel adapter

Without an adapter:

```text
Device → Socket A
Device → Socket B
Device → Socket C
```

With a universal adapter:

```text
              ┌→ Socket A
Device → Adapter → Socket B
              └→ Socket C
```

For us:

```text
                ┌→ Gemini
Python → LangChain
                └→ Groq
```

LangChain does **not** make Gemini and Groq the same model. It makes our application code more consistent at the layer where we interact with them.

In [10]:
# ==========================================================
# CELL 9: IMPORT LANGCHAIN MODEL INTEGRATIONS
# ==========================================================

# LangChain integration for Groq-hosted chat models.
from langchain_groq import ChatGroq

# LangChain integration for Google's Gemini chat models.
from langchain_google_genai import ChatGoogleGenerativeAI

print("LangChain integrations imported.")

LangChain integrations imported.


In [11]:
# ==========================================================
# CELL 10: CREATE BOTH LANGCHAIN MODEL OBJECTS
# ==========================================================
#
# We still create provider-specific model objects because
# Gemini and Groq are different services.
#
# The benefit appears after this setup: both support the
# LangChain .invoke(...) pattern.
# ==========================================================

groq_llm = ChatGroq(
    model=GROQ_MODEL,
    api_key=GROQ_API_KEY,
)

gemini_llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    api_key=GEMINI_API_KEY,
)

print("Groq LangChain model ready.")
print("Gemini LangChain model ready.")

Groq LangChain model ready.
Gemini LangChain model ready.


# Part 6 — Understand `invoke()` and `.content`

## `invoke()`

```python
response = llm.invoke(question)
```

For today's lesson, read `invoke()` as:

> **Send this input through this LangChain component and return one result.**

## What comes back?

A LangChain chat model typically returns a message object such as an:

```text
AIMessage
```

It can contain the answer plus metadata.

## `.content`

For our text examples:

```python
response.content
```

gives us the generated answer.

Mental model:

```text
question
   ↓
llm.invoke(question)
   ↓
AIMessage
   ↓
.content
   ↓
generated text
```

In [12]:
# ==========================================================
# CELL 11: ASK GROQ THROUGH LANGCHAIN
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

groq_lc_response = groq_llm.invoke(question)

print(groq_lc_response.content)

**What is an API?**

Think of an API (Application Programming Interface) like a *menu at a restaurant*.

| Restaurant | Computer Programs |
|------------|-------------------|
| The menu lists all the dishes you can order. | An API lists all the things a program can *ask* another program to do. |
| You tell the waiter what you want, and the kitchen makes it. | You send a request to the API, and the other program does something (or gives you data) in return. |
| The waiter knows how to translate your order into the kitchen’s language. | The API is the “translator” that lets two different programs talk even if they’re built in different ways. |

---

### A Simple Example

Imagine you’re playing a game on your phone that shows the weather outside. The game itself doesn’t have a built‑in weather sensor, but it can ask a *weather service* for the current temperature.

1. **You (the game)**: “Hey, what’s the weather in my city?”
2. **API**: Sends that question to the weather service.
3. **Wea

In [13]:
# ==========================================================
# CELL 12: ASK GEMINI THROUGH LANGCHAIN
# ==========================================================

question = "Explain what an API is to a 12-year-old student."

gemini_lc_response = gemini_llm.invoke(question)

print(gemini_lc_response.content)

[{'type': 'text', 'text': 'Imagine you are at a cool restaurant. \n\nYou (the **User**) are sitting at your table, looking at the menu. You want a cheeseburger, but the kitchen (the **System**) is way in the back, behind closed doors, and you aren\'t allowed to go in there. \n\nHow do you get your cheeseburger? \n\nYou need **The Waiter**. \n\nYou tell the waiter what you want. The waiter walks back to the kitchen, gives your order to the chef, waits for the chef to cook the burger, and then brings the burger back out to your table.\n\nIn the digital world, an **API** (which stands for *Application Programming Interface*) is **The Waiter**. \n\nHere is how it works with apps and websites:\n\nLet’s say you are playing a video game on your phone and you want to log in using your Google account. \n* Your game is **you** sitting at the table.\n* Google’s massive computer system is the **kitchen** in the back.\n* The **API** is the waiter that runs between the game and Google, asking, *"Hey

# Stop and compare

## Direct SDKs

Gemini:

```python
response = gemini_client.models.generate_content(...)
print(response.text)
```

Groq:

```python
response = groq_client.chat.completions.create(...)
print(response.choices[0].message.content)
```

## LangChain

Gemini:

```python
response = gemini_llm.invoke(question)
print(response.content)
```

Groq:

```python
response = groq_llm.invoke(question)
print(response.content)
```

The underlying providers are different.

The **application-level calling pattern** is now much more consistent.

In [14]:
# ==========================================================
# CELL 13: LOOK INSIDE AN AIMessage
# ==========================================================

response = groq_llm.invoke("Say hello in one short sentence.")

print("Python type:")
print(type(response))

print("\nWhole response object:")
print(response)

print("\nOnly the generated content:")
print(response.content)

Python type:
<class 'langchain_core.messages.ai.AIMessage'>

Whole response object:
content='Hello!' additional_kwargs={'reasoning_content': 'User says: "Say hello in one short sentence." They want a short sentence saying hello. Probably "Hello!" or "Hi there!" or "Hey!" The instruction: "Say hello in one short sentence." So respond with a short sentence. Probably "Hello!" is best.'} response_metadata={'token_usage': {'completion_tokens': 69, 'prompt_tokens': 78, 'total_tokens': 147, 'completion_time': 0.070978668, 'completion_tokens_details': {'reasoning_tokens': 58}, 'prompt_time': 0.003729898, 'prompt_tokens_details': None, 'queue_time': 0.385560145, 'total_time': 0.074708566}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_e23fc997ca', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0b59f-6f6e-79a1-b57a-db819c0e8952-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 78, 'output_token

For now, focus on:

```python
response.content
```

Later, the rest of the message object can be useful for things such as:

- token usage
- metadata
- debugging
- tool calls
- tracing

# Part 7 — Switch providers with one helper function

We can keep model objects in a dictionary:

```python
models = {
    "groq": groq_llm,
    "gemini": gemini_llm,
}
```

Then our higher-level code can choose a provider while still using:

```text
.invoke(...)
.content
```

In [15]:
# ==========================================================
# CELL 14: ONE SIMPLE INTERFACE FOR BOTH MODELS
# ==========================================================

models = {
    "groq": groq_llm,
    "gemini": gemini_llm,
}


def ask_llm(provider, question):
    """Ask one of our LangChain chat models a question."""

    # Get the selected model object.
    llm = models[provider]

    # Both LangChain chat models support invoke().
    response = llm.invoke(question)

    # For this lesson, return the generated text.
    return response.content


print("Reusable ask_llm() function created.")

Reusable ask_llm() function created.


In [16]:
# ==========================================================
# CELL 15: SAME FUNCTION — GROQ
# ==========================================================

answer = ask_llm(
    provider="groq",
    question="What is machine learning? Explain in 4 simple lines.",
)

print(answer)

Machine learning is a way for computers to learn from data instead of being explicitly programmed.  
It finds patterns, rules, or models that explain the data.  
Once trained, the model can make predictions or decisions on new, unseen data.  
Essentially, it lets machines improve automatically by learning from experience.


In [ ]:
# ==========================================================
# CELL 16: SAME FUNCTION — GEMINI
# ==========================================================

answer = ask_llm(
    provider="gemini",
    question="What is machine learning? Explain in 4 simple lines.",
)

print(answer)  #for structured output of gemini print(answer[0]["text"])     From that first dictionary, give me only the value stored under "text"

[{'type': 'text', 'text': '1. Machine learning is a branch of artificial intelligence that lets computers learn from data without being explicitly programmed.\n2. Instead of following strict rules, it finds patterns and makes decisions by analyzing large amounts of examples.\n3. The more data it processes, the better and more accurate its predictions become over time.\n4. Common examples include Netflix recommendations, spam email filters, and voice assistants like Siri.', 'extras': {'signature': 'El4KXAFpFH0TaihdRDvQogwlBGfbEqW+cyigzw8gG4mnPlGxVP1tzTVug+GvFIHvOZDF4X3dRuvfvjrws5PfHO++u12ZkxcjT9/NMivFtbi/M/PBvUfhnNiEj5RqUJum'}}]


Only the provider changed:

```python
provider="groq"
```

versus:

```python
provider="gemini"
```

This is an example of **abstraction**: hiding lower-level differences behind a simpler common interface.

# Part 8 — Practice with NORMAL prompts

A normal prompt can simply be whatever the user types.

We will use Python's:

```python
input()
```

so students can ask different questions without editing the rest of the code.

In [25]:
# ==========================================================
# CELL 17: INTERACTIVE QUESTION
# ==========================================================

provider = input("Choose provider (groq/gemini): ").strip().lower()
question = input("Ask your question: ")

if provider not in models:
    print("Please choose either 'groq' or 'gemini'.")
else:
    answer = ask_llm(provider, question)

    print("\n--- LLM ANSWER ---\n")
    print(answer)


--- LLM ANSWER ---

The date today is **2026‑09‑18**.


## Student practice

Run the previous cell several times.

Try:

```text
What is an LLM?
```

```text
Explain tokens in simple words.
```

```text
What is the difference between AI and machine learning?
```

```text
Explain a Python function using a real-life analogy.
```

Ask the **same question** to both providers.

Do they answer exactly the same way? Why might different models respond differently even when the prompt is identical?

# Part 9 — Fixed / static prompts

So far, the user controlled the question.

Applications can also contain instructions written by the developer.

Example:

```text
You are a beginner-friendly AI teacher.
Explain what an API is.
Use simple English.
Use one everyday analogy.
Keep the answer under 150 words.
```

We can store this instruction in a Python variable and send the same prompt to either model.

In [26]:
# ==========================================================
# CELL 18: CREATE A FIXED / STATIC PROMPT
# ==========================================================

STATIC_PROMPT = """
You are a beginner-friendly AI teacher.

Explain what an API is.

Rules:
- Use simple English.
- Assume the student has never programmed before.
- Give one everyday analogy.
- Keep the answer under 150 words.
"""

print(STATIC_PROMPT)


You are a beginner-friendly AI teacher.

Explain what an API is.

Rules:
- Use simple English.
- Assume the student has never programmed before.
- Give one everyday analogy.
- Keep the answer under 150 words.



In [27]:
# ==========================================================
# CELL 19: SAME STATIC PROMPT → GROQ
# ==========================================================

response = groq_llm.invoke(STATIC_PROMPT)
print(response.content)

An **API** (Application Programming Interface) is a set of rules that lets one computer program talk to another.

**Everyday analogy:**  
Think of a restaurant menu. The menu lists all the dishes you can order and shows you what each dish contains. You don’t need to know how the kitchen cooks the food; you just tell the waiter what you want, and the waiter brings it back.  
The menu is the API: it tells you what you can ask for (the dishes) and how to ask (the order format). The kitchen is the program that actually prepares the food.

In the same way, an API lets you request data or services from a program without needing to understand its internal code. It keeps the “menu” simple so everyone can use it.


In [ ]:
# ==========================================================
# CELL 20: SAME STATIC PROMPT → GEMINI
# ==========================================================

response = gemini_llm.invoke(STATIC_PROMPT)
print(response.content) #[0]["text"]

Imagine you are sitting at a restaurant table. You look at the menu, but you cannot walk into the kitchen to cook your own food. You need a **waiter** to take your order to the kitchen, and then bring the food back to your table.

An **API** (Application Programming Interface) is just like that waiter in the digital world. 

When you use an app on your phone to check the weather, the app doesn't know the weather. It uses an API to send a request to a giant weather computer, and the API brings the weather report back to your phone. 

In short, an API is a messenger that lets different apps talk to each other and share information!


# Compare the responses

Ask yourself:

- Did both use an analogy?
- Did both stay beginner-friendly?
- Did both follow the word limit?
- What would happen if we removed the rules?
- What would happen if we changed the audience?

This is our first clue that **the wording and structure of the prompt matter**.

# Part 10 — One step toward reusable prompts

What if the teaching style stays the same but the **topic changes**?

Instead of copying the whole prompt repeatedly, we can create a template containing:

```text
{topic}
```

Then `{topic}` can become APIs, tokens, embeddings, transformers, Python functions, or anything else.

In [32]:
# ==========================================================
# CELL 21: CREATE A CHAT PROMPT TEMPLATE
# ==========================================================

from langchain_core.prompts import ChatPromptTemplate

# {topic} is a variable / placeholder.
prompt_template = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are a beginner-friendly AI teacher.
            The student has no prior programming knowledge.
            Use simple English and practical analogies.
            """,
        ),
        (
            "human",
            """
            Explain {topic}.

            Requirements:
            - Keep it under 150 words.
            - Give one everyday analogy.
            - Give one simple example.
            """,
        ),
    ]
)

print("Prompt template created.")

Prompt template created.


## Why `system` and `human`?

### `system`

High-level behavior/instructions for the assistant.

Example:

```text
You are a beginner-friendly AI teacher.
```

### `human`

The user's actual request.

Example:

```text
Explain APIs.
```

We will study instruction design much more deeply in the next class.

# Part 11 — Build a simple LangChain chain

LangChain uses the pipe symbol:

```python
|
```

to compose compatible components.

```python
chain = prompt_template | groq_llm
```

Read it left to right:

```text
input variables
      ↓
prompt template
      ↓
     LLM
      ↓
   AIMessage
```

In [33]:
# ==========================================================
# CELL 22: PROMPT TEMPLATE → GROQ
# ==========================================================

groq_chain = prompt_template | groq_llm

# The template expects a variable named {topic}, so invoke()
# receives a dictionary containing the value for "topic".
response = groq_chain.invoke(
    {
        "topic": "tokens in generative AI"
    }
)

print(response.content)

**Tokens in generative AI**

Think of a token as a *building block* that the AI uses to understand and produce text. It’s not always a whole word; sometimes it’s part of a word or even a punctuation mark.  

**Everyday analogy**  
Imagine a sentence is a sentence made of LEGO bricks. Each brick (token) is a small piece that the AI snaps together to build the whole sentence.

**Simple example**  
The sentence “I love pizza.” might be split into tokens like: “I”, “love”, “pizza”, “.”. The AI looks at these tokens, learns how they fit together, and then can create new sentences by combining new tokens in the same way.


In [34]:
# ==========================================================
# CELL 23: SAME PROMPT TEMPLATE → GEMINI
# ==========================================================

gemini_chain = prompt_template | gemini_llm

response = gemini_chain.invoke(
    {
        "topic": "tokens in generative AI"
    }
)

print(response.content)

[{'type': 'text', 'text': 'Imagine you are building with **Lego blocks**, but instead of plastic bricks, you are building with pieces of words! \n\nIn Generative AI, **tokens** are these little building blocks. An AI doesn\'t read whole sentences like we do. It chops text up into tiny chunks called tokens. A token can be a whole word, part of a word, or even just a punctuation mark. \n\n*   **Everyday Analogy:** Think of tokens like **syllables** or **puzzle pieces**. You cannot build a puzzle without putting the small pieces together first. \n\n*   **Simple Example:** The sentence *"I love cats!"* might be chopped into 4 tokens by the AI: \n    1. "I" \n    2. " love" \n    3. " cat" \n    4. "s!"\n\nWhen the AI talks to you, it predicts the very next token, one Lego block at a time, until the whole message is built!', 'extras': {'signature': 'El4KXAFpFH0Tog0klw5ngTEdqzYqBE+DuF4Z6oG9T/eyuYqCKY59LRd0WCBNyPbwpgAwMjcsESWwGoE99F/5uD/tNQf7/DmZjo284aKKsuDnmoBWd7xeL3T1Jiwsw2Xu'}}]


Notice:

```python
prompt_template | groq_llm
```

and:

```python
prompt_template | gemini_llm
```

The prompt structure stayed the same. Only the model object changed.

In [35]:
# ==========================================================
# CELL 24: REUSE THE SAME CHAIN WITH A NEW TOPIC
# ==========================================================

response = groq_chain.invoke(
    {
        "topic": "embeddings"
    }
)

print(response.content)

**Embeddings** are a way to turn things (words, images, sounds) into numbers that a computer can understand.  
Think of a **library map**: each book has a spot on the map. Books that are about similar topics sit near each other, so a computer can “see” their closeness.  
**Simple example**: The word “cat” might be represented by the numbers \[0.2, 0.8, 0.1\]. The word “dog” could be \[0.25, 0.75, 0.12\]. Because the numbers are similar, the computer knows cats and dogs are related. Embeddings let computers find patterns and similarities without reading the whole word or picture.


In [36]:
# ==========================================================
# CELL 25: STUDENT PRACTICE
# ==========================================================
#
# Change ONLY the topic.
# Suggested topics:
# - transformers
# - context windows
# - API
# - machine learning
# - virtual environments
# - Python functions
# ==========================================================

my_topic = "transformers"

response = gemini_chain.invoke(
    {
        "topic": my_topic
    }
)

print(response.content)

[{'type': 'text', 'text': 'Imagine you are reading a long mystery novel. Instead of looking at just one word at a time, you look at the whole sentence to understand the secret meaning. That is what an AI **Transformer** does! \n\nIt is a special computer brain that looks at all the words in a sentence *at the same time* and figures out how they connect. \n\n**Analogy:** Think of a group of friends planning a surprise party. Instead of passing notes one by one, they sit in a circle and talk together, paying attention to who is speaking loudest about the cake, the balloons, and the guest of honor. \n\n**Example:** In the sentence, *"The dog chased the cat because **it** was scared,"* a Transformer easily knows that "**it**" means the cat, not the dog, by looking at the whole picture at once!', 'extras': {'signature': 'El4KXAFpFH0TwCJnkltFm7KpOMEl51Fz89nstcf46mPWxpErpu1FlomF3byATU7zl+y6kI5iojo7vQPYSF/THvmslRNsk2BqBcbG+a0H3kzzMYMvds5xCB5zMPEnMMrE'}}]


# Part 12 — Final experiment: why does prompt wording matter?

Do **not** memorize prompt-engineering rules yet.

Just observe what happens when the **model stays the same** but the prompt changes.

In [37]:
# ==========================================================
# CELL 26: VAGUE PROMPT
# ==========================================================

prompt_a = "Explain APIs."

response_a = groq_llm.invoke(prompt_a)

print(response_a.content)

## What Is an API?

**API** stands for **Application Programming Interface**.  
It’s a set of rules, protocols, and tools that lets one piece of software talk to another. Think of it as a menu in a restaurant:

- The **menu** lists dishes (functions/operations) you can order.
- The **waiter** (the API) takes your order and brings you the food (data/response).
- You never need to know how the kitchen prepares the dish; you just need to know the menu and how to place an order.

In software terms, an API defines **what** you can do (methods, endpoints) and **how** to do it (parameters, formats, authentication), but it hides the underlying implementation.

---

## Core Concepts

| Concept | What It Means | Example |
|---------|---------------|---------|
| **Endpoint** | A specific URL (or function) you hit to perform an action. | `https://api.github.com/users/octocat` |
| **Method** | The type of HTTP verb (REST) or operation (SOAP/GraphQL) you use. | `GET`, `POST`, `PUT`, `DELETE` |
| **R

In [38]:
# ==========================================================
# CELL 27: MORE SPECIFIC PROMPT
# ==========================================================

prompt_b = """
You are teaching a 15-year-old student who has never programmed before.

Explain what an API is.

Requirements:
1. Use less than 150 words.
2. Start with a one-sentence definition.
3. Use a restaurant waiter analogy.
4. Give one software example.
5. Finish with one question to test the student's understanding.
"""

response_b = groq_llm.invoke(prompt_b)

print(response_b.content)

An API is a set of rules that lets one piece of software talk to another.  
Imagine a waiter in a restaurant who takes your order, tells the kitchen, and brings back your food. The waiter is like an API: you ask for something, the waiter passes your request to the kitchen, and returns the result.  
For example, the Google Maps API lets your app ask for directions and get them back.  

If you wanted a weather report, what would you use the API for?


# Think before the next class

Compare `prompt_a` and `prompt_b`.

1. Which response was easier for a beginner to understand?
2. Which response had a more predictable structure?
3. Why did the model behave differently?
4. Which words in the prompt influenced the output?
5. Can we intentionally design prompts to get more useful and reliable responses?

Do not worry if you cannot answer all of these yet.

That is exactly where the next lesson begins:

# Next Class — Prompt Engineering

We will move from:

```text
"Ask the LLM something"
```

toward:

```text
"Design the instruction so the LLM understands
its role, task, context, constraints, examples,
and expected output."
```

# End-of-Class Challenge

Use the **same model** and the same topic:

```text
Machine Learning
```

Create three prompts.

### Prompt 1 — Very simple

```text
Explain machine learning.
```

### Prompt 2 — Add the audience

Tell the model who it is teaching.

### Prompt 3 — Add structure

Include:

- a role
- audience
- output length
- analogy requirement
- one example
- a requested output structure

Compare all three responses.

### Final question

> If the model stayed the same, but response quality changed when the prompt changed, what exactly are we programming?

Bring your answer to the next class.

# Quick Revision

## Direct Gemini

```python
response = gemini_client.models.generate_content(...)
print(response.text)
```

## Direct Groq

```python
response = groq_client.chat.completions.create(...)
print(response.choices[0].message.content)
```

## LangChain Gemini

```python
response = gemini_llm.invoke(question)
print(response.content)
```

## LangChain Groq

```python
response = groq_llm.invoke(question)
print(response.content)
```

## Reusable prompt chain

```python
chain = prompt_template | llm
response = chain.invoke({"topic": "APIs"})
print(response.content)
```

## The journey

```text
Different provider APIs
        ↓
Provider-specific integration differences
        ↓
LangChain abstraction
        ↓
Common invoke/content pattern
        ↓
Reusable prompt structures
        ↓
PROMPT ENGINEERING
```